In [26]:
!pip install flask flask_login validators pyngrok

In [27]:
%%bash
cat > app.py <<'PY'
from flask import Flask, render_template, request, redirect, url_for, flash
from flask_login import LoginManager, UserMixin, login_user, login_required, logout_user, current_user
import sqlite3
import string
import random
import validators

app = Flask(__name__)
app.secret_key = "secret-key"

login_manager = LoginManager()
login_manager.init_app(app)
login_manager.login_view = "login"

# ---------------- DATABASE ----------------
def get_db():
    conn = sqlite3.connect("urls.db")
    conn.row_factory = sqlite3.Row
    return conn

def create_tables():
    conn = get_db()
    conn.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            password TEXT NOT NULL
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS urls (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER,
            original_url TEXT NOT NULL,
            short_code TEXT UNIQUE NOT NULL,
            FOREIGN KEY (user_id) REFERENCES users(id)
        )
    """)
    conn.commit()
    conn.close()

create_tables()

# ---------------- USER CLASS ----------------
class User(UserMixin):
    def __init__(self, id, username, password):
        self.id = id
        self.username = username
        self.password = password

@login_manager.user_loader
def load_user(user_id):
    conn = get_db()
    user = conn.execute("SELECT * FROM users WHERE id = ?", (user_id,)).fetchone()
    conn.close()
    if user:
        return User(user["id"], user["username"], user["password"])
    return None

# ---------------- SHORT CODE GENERATOR ----------------
def generate_short_code(length=6):
    chars = string.ascii_letters + string.digits
    return ''.join(random.choice(chars) for _ in range(length))

# ---------------- ROUTES ----------------
@app.route("/")
def home():
    return redirect(url_for("login"))

@app.route("/signup", methods=["GET", "POST"])
def signup():
    if request.method == "POST":
        username = request.form["username"]
        password = request.form["password"]

        if len(username) < 5 or len(username) > 9:
            flash("Username must be between 5 to 9 characters long", "danger")
            return redirect(url_for("signup"))

        conn = get_db()
        existing = conn.execute("SELECT * FROM users WHERE username = ?", (username,)).fetchone()

        if existing:
            flash("This username already exists…", "danger")
            conn.close()
            return redirect(url_for("signup"))

        conn.execute("INSERT INTO users (username, password) VALUES (?, ?)", (username, password))
        conn.commit()
        conn.close()
        flash("Signup successful! Please login.", "success")
        return redirect(url_for("login"))

    return render_template("signup.html")

@app.route("/login", methods=["GET", "POST"])
def login():
    if request.method == "POST":
        username = request.form["username"]
        password = request.form["password"]

        conn = get_db()
        user = conn.execute("SELECT * FROM users WHERE username = ? AND password = ?", (username, password)).fetchone()
        conn.close()

        if user:
            user_obj = User(user["id"], user["username"], user["password"])
            login_user(user_obj)
            return redirect(url_for("dashboard"))
        else:
            flash("Invalid username or password", "danger")

    return render_template("login.html")

@app.route("/dashboard", methods=["GET", "POST"])
@login_required
def dashboard():
    short_url = None
    error = None

    if request.method == "POST":
        original_url = request.form["url"]

        if not validators.url(original_url):
            error = "Invalid URL. Please enter a valid URL."
        else:
            short_code = generate_short_code()

            conn = get_db()
            conn.execute(
                "INSERT INTO urls (user_id, original_url, short_code) VALUES (?, ?, ?)",
                (current_user.id, original_url, short_code)
            )
            conn.commit()
            conn.close()

            short_url = request.host_url + short_code

    return render_template("dashboard.html", short_url=short_url, error=error)

@app.route("/history")
@login_required
def history():
    conn = get_db()
    urls = conn.execute("SELECT * FROM urls WHERE user_id = ?", (current_user.id,)).fetchall()
    conn.close()
    return render_template("history.html", urls=urls)

@app.route("/<short_code>")
def redirect_url(short_code):
    conn = get_db()
    result = conn.execute("SELECT original_url FROM urls WHERE short_code = ?", (short_code,)).fetchone()
    conn.close()

    if result:
        return redirect(result["original_url"])
    else:
        return "URL Not Found", 404

@app.route("/logout")
@login_required
def logout():
    logout_user()
    return redirect(url_for("login"))

if __name__ == "__main__":
    app.run(debug=True)
PY

In [28]:
%%bash
mkdir -p templates
cat > templates/signup.html <<'HTML'
<!DOCTYPE html>
<html>
<head>
    <title>Signup</title>
    <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css">
</head>
<body class="container mt-5">

<h2>Signup</h2>
<form method="POST">
    <input type="text" name="username" class="form-control" placeholder="Username" required>
    <input type="password" name="password" class="form-control mt-3" placeholder="Password" required>
    <button class="btn btn-primary mt-3">Signup</button>
</form>

<a href="/login" class="btn btn-link mt-3">Already have an account? Login</a>

{% with messages = get_flashed_messages(with_categories=true) %}
  {% if messages %}
    {% for category, msg in messages %}
      <div class="alert alert-{{ category }} mt-3">{{ msg }}</div>
    {% endfor %}
  {% endif %}
{% endwith %}

</body>
</html>
HTML

In [29]:
%%bash
cat > templates/login.html <<'HTML'
<!DOCTYPE html>
<html>
<head>
    <title>Login</title>
    <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css">
</head>
<body class="container mt-5">

<h2>Login</h2>
<form method="POST">
    <input type="text" name="username" class="form-control" placeholder="Username" required>
    <input type="password" name="password" class="form-control mt-3" placeholder="Password" required>
    <button class="btn btn-primary mt-3">Login</button>
</form>

<a href="/signup" class="btn btn-link mt-3">Create new account</a>

{% with messages = get_flashed_messages(with_categories=true) %}
  {% if messages %}
    {% for category, msg in messages %}
      <div class="alert alert-{{ category }} mt-3">{{ msg }}</div>
    {% endfor %}
  {% endif %}
{% endwith %}

</body>
</html>
HTML

In [30]:
%%bash
cat > templates/dashboard.html <<'HTML'
<!DOCTYPE html>
<html>
<head>
    <title>Dashboard</title>
    <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css">
</head>
<body class="container mt-5">

<h2>URL Shortener Dashboard</h2>
<p>Logged in as: {{ current_user.username }}</p>
<a href="/logout" class="btn btn-danger">Logout</a>

<form method="POST">
    <input type="text" name="url" class="form-control" placeholder="Enter URL" required>
    <button class="btn btn-primary mt-3">Shorten URL</button>
</form>

{% if error %}
<div class="alert alert-danger mt-3">{{ error }}</div>
{% endif %}

{% if short_url %}
<div class="mt-3">
    <input type="text" id="shortUrl" class="form-control" value="{{ short_url }}" readonly>
    <button onclick="copyText()" class="btn btn-success mt-2">Copy</button>
</div>
{% endif %}

<a href="/history" class="btn btn-link mt-4">View My History</a>

<script>
function copyText() {
    let copyInput = document.getElementById("shortUrl");
    copyInput.select();
    copyInput.setSelectionRange(0, 99999);
    document.execCommand("copy");
    alert("Copied!");
}
</script>

</body>
</html>
HTML

In [31]:
%%bash
cat > templates/history.html <<'HTML'
<!DOCTYPE html>
<html>
<head>
    <title>History</title>
    <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css">
</head>
<body class="container mt-5">

<h2>My URL History</h2>
<a href="/dashboard" class="btn btn-primary">Back</a>

<table class="table table-bordered mt-3">
    <thead>
        <tr>
            <th>Original URL</th>
            <th>Short URL</th>
        </tr>
    </thead>
    <tbody>
        {% for url in urls %}
        <tr>
            <td>{{ url["original_url"] }}</td>
            <td>{{ request.host_url }}{{ url["short_code"] }}</td>
        </tr>
        {% endfor %}
    </tbody>
</table>

</body>
</html>
HTML

In [39]:
from pyngrok import ngrok
import subprocess

# Set your ngrok authtoken here. Replace 'YOUR_NGROK_AUTH_TOKEN' with your actual token.
# You can get one from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("26p5O1vfFGLsdf9f8D3dzHAcmUv_454E7DaBz26mgs9yVkzQi")

# Start ngrok tunnel
public_url = ngrok.connect(5000).public_url
print("Public URL:", public_url)

# Run Flask app
subprocess.Popen(["python", "app.py"])

PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: Your account may not run more than 3 endpoints over a single ngrok agent session.\nThe endpoints already running on this session are:\ntn_38PuPEDFsLEI1zBrOclnarXwP9N, tn_38PuZPM0zPk6SAZ1ZFaA6IDbyHS, tn_38PwGUZCbtyQ5PPeb2Bupxf4AE3.\nUpgrade to a Pay-as-you-go plan at: https://dashboard.ngrok.com/billing/choose-a-plan?plan=paygo\r\n\r\nERR_NGROK_324\r\n"}}
